In [ ]:
# Import packages
import pandas as pd 

In [ ]:
# Load NASS Corn Acreage Data
df = pd.read_csv("../data/raw-data/usda_nass_corn.csv")
df.head()

,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,SURVEY,2012,YEAR,NaN,COUNTY,ALABAMA,1,BLACK BELT,40,BULLOCK,...,NaN,NaN,0,NaN,CORN,CORN - ACRES PLANTED,TOTAL,NOT SPECIFIED,"1,300",NaN
1,SURVEY,2012,YEAR,NaN,COUNTY,ALABAMA,1,BLACK BELT,40,DALLAS,...,NaN,NaN,0,NaN,CORN,CORN - ACRES PLANTED,TOTAL,NOT SPECIFIED,"5,000",NaN
2,SURVEY,2012,YEAR,NaN,COUNTY,ALABAMA,1,BLACK BELT,40,ELMORE,...,NaN,NaN,0,NaN,CORN,CORN - ACRES PLANTED,TOTAL,NOT SPECIFIED,"3,000",NaN
3,SURVEY,2012,YEAR,NaN,COUNTY,ALABAMA,1,BLACK BELT,40,HALE,...,NaN,NaN,0,NaN,CORN,CORN - ACRES PLANTED,TOTAL,NOT SPECIFIED,"1,000",NaN
4,SURVEY,2012,YEAR,NaN,COUNTY,ALABAMA,1,BLACK BELT,40,OTHER (COMBINED) COUNTIES,...,NaN,NaN,0,NaN,CORN,CORN - ACRES PLANTED,TOTAL,NOT SPECIFIED,"10,200",NaN


In [8]:
### Clean data
# Drop "Other Combined Counties" rows 
# These rows have no County ANSI code and can't be joined to a shapefile.
df = df[df['County ANSI'].notna()].copy()

# Build 5-digit FIPS code 
df['state_fips'] = df['State ANSI'].astype(int).astype(str).str.zfill(2)
df['county_fips'] = df['County ANSI'].astype(int).astype(str).str.zfill(3)
df['FIPS'] = df['state_fips'] + df['county_fips']

# Convert value column to float
df['acres'] = df['Value'].str.replace(',', '', regex=False).astype(float)

# Keep only the columns we need 
df_clean = df[['FIPS', 'State', 'County', 'Year', 'acres']].copy()

# Pivot to wide format 
pivot = df_clean.pivot_table(
    index=['FIPS', 'State', 'County'],
    columns='Year',
    values='acres',
    aggfunc='sum'   # in case of any duplicate rows for the same county/year
)

# Rename year columns to something readable
pivot.columns.name = None
pivot = pivot.rename(columns={2006: 'acres_2006', 2012: 'acres_2012'})
pivot = pivot.reset_index()

# Drop counties missing either year 
pivot = pivot.dropna(subset=['acres_2006', 'acres_2012'])

# Calculate change metrics 
pivot['change_acres'] = pivot['acres_2012'] - pivot['acres_2006']
pivot['pct_change']   = (pivot['change_acres'] / pivot['acres_2006']) * 100

# Preview and save 
print(f"Counties retained: {len(pivot)}")
print()
print(pivot.head(10).to_string(index=False))
print()
print("Summary statistics:")
print(pivot[['acres_2006', 'acres_2012', 'change_acres', 'pct_change']].describe().round(1))

pivot.to_csv('../data/processed-data/corn_acreage_cleaned.csv', index=False)
print("\nSaved to corn_acreage_cleaned.csv")

Counties retained: 1573

 FIPS   State   County  acres_2006  acres_2012  change_acres  pct_change
01003 ALABAMA  BALDWIN      2100.0      3100.0        1000.0   47.619048
01015 ALABAMA  CALHOUN      1000.0      1600.0         600.0   60.000000
01019 ALABAMA CHEROKEE      1800.0      4800.0        3000.0  166.666667
01031 ALABAMA   COFFEE      6500.0      3400.0       -3100.0  -47.692308
01033 ALABAMA  COLBERT     11500.0     24500.0       13000.0  113.043478
01035 ALABAMA  CONECUH      1800.0       800.0       -1000.0  -55.555556
01043 ALABAMA  CULLMAN      3300.0      3500.0         200.0    6.060606
01045 ALABAMA     DALE      4000.0      2200.0       -1800.0  -45.000000
01047 ALABAMA   DALLAS      3600.0      5000.0        1400.0   38.888889
01049 ALABAMA  DE KALB     10600.0     12700.0        2100.0   19.811321

Summary statistics:
       acres_2006  acres_2012  change_acres  pct_change
count      1573.0      1573.0        1573.0      1573.0
mean      47613.1     58760.5       111